In [0]:
# =====================================
# NOTEBOOK : NB_SILVER_PROVIDER
# LAYER    : SILVER
# TARGET   : ODS_PROVIDER
# =====================================


In [0]:
%run ../COMMON/NB_CONFIG

In [0]:
%run ../COMMON/NB_CONNECTION

In [0]:
%run ../COMMON/NB_UTILS

In [0]:
provider_df = spark.read.jdbc(
    url=srcJdbcUrl,
    table="BRONZE.PROVIDER",
    properties=connectionProperties
)

provider_address_df = spark.read.jdbc(
    url=srcJdbcUrl,
    table="BRONZE.PROVIDER_ADDRESS",
    properties=connectionProperties
)

print("PROVIDER COUNT :", provider_df.count())
print("PROVIDER ADDRESS COUNT:", provider_address_df.count())


In [0]:
null_count = provider_df.filter(F.col("PROVIDER_ID").isNull()).count()
if null_count > 0:
    raise Exception(f"NULL PROVIDER_ID FOUND : {null_count}")


In [0]:
joined_df = (
    provider_df.alias("p")
    .join(
        provider_address_df.alias("a"),
        F.col("p.PROVIDER_ID") == F.col("a.PROVIDER_ID"),
        "left"
    )
)


In [0]:
ods_provider_df = joined_df.select(
    F.col("p.PROVIDER_ID"),
    F.col("p.PROVIDER_NAME"),
    F.col("p.SPECIALITY"),
    F.col("p.PHONE"),
    F.col("p.EMAIL"),
    F.col("p.STATUS"),

    F.col("a.ADDRESS_LINE1"),
    F.col("a.ADDRESS_LINE2"),
    F.col("a.CITY"),
    F.col("a.STATE"),
    F.col("a.ZIP_CODE"),
    F.col("a.COUNTRY"),

    F.col("p.SRC_CREATED_DATETIME")
)


In [0]:
ods_provider_df = (
    ods_provider_df
    .withColumn("PROVIDER_NAME", F.initcap(F.col("PROVIDER_NAME")))
)


In [0]:
ods_provider_df = ods_provider_df.fillna({
    "CITY":"UNKNOWN",
    "STATE":"UNKNOWN",
    "COUNTRY":"INDIA",
    "ZIP_CODE":"000000"
})


In [0]:
window_spec = Window.partitionBy("PROVIDER_ID").orderBy(F.col("SRC_CREATED_DATETIME").desc())

ods_provider_df = (
    ods_provider_df
    .withColumn("RN", F.row_number().over(window_spec))
    .filter(F.col("RN") == 1)
    .drop("RN")
)


In [0]:
ods_provider_df = ods_provider_df.withColumn(
    "HASH_VALUE",
    F.sha2(
        F.concat_ws("|",
            F.col("PROVIDER_NAME"),
            F.col("SPECIALITY"),
            F.col("PHONE"),
            F.col("EMAIL"),
            F.col("STATUS"),
            F.col("ADDRESS_LINE1"),
            F.col("CITY"),
            F.col("STATE"),
            F.col("ZIP_CODE"),
            F.col("COUNTRY")
        ),
        256
    )
)


In [0]:
ods_provider_df = (
    ods_provider_df
    .withColumn("LOAD_DATE", F.current_timestamp())
    .withColumn("IS_CURRENT", F.lit("Y"))
    .withColumn("EFFECTIVE_DATE", F.current_timestamp())
    .withColumn("END_DATE", F.lit("9999-12-31").cast("timestamp"))
)


In [0]:
target_count = spark.table(ods_provider_table).count()
print(f"TARGET RECORD COUNT : {target_count}")

if target_count == 0:
    # Initial full load
    ods_provider_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(ods_provider_table)

    print("INITIAL LOAD COMPLETED")
    dbutils.notebook.exit("Initial load done")

else:
    print("SCD2 LOAD STARTED")
    target_df = spark.table(ods_provider_table)
    current_df = target_df.filter(F.col("IS_CURRENT") == "Y")
    print(f"CURRENT ACTIVE RECORDS : {current_df.count()}")


In [0]:
changed_df = (
    ods_provider_df.alias("s")
    .join(current_df.alias("t"), "PROVIDER_ID")
    .filter(F.col("s.HASH_VALUE") != F.col("t.HASH_VALUE"))
    .select("s.*")
)


In [0]:
new_provider_df = (
    ods_provider_df.alias("s")
    .join(current_df.alias("t"), "PROVIDER_ID", "left_anti")
)


In [0]:
delta_table = DeltaTable.forName(spark, ods_provider_table)

expire_df = changed_df.select("PROVIDER_ID").distinct()

delta_table.alias("t").merge(
    expire_df.alias("s"),
    "t.PROVIDER_ID = s.PROVIDER_ID AND t.IS_CURRENT = 'Y'"
).whenMatchedUpdate(
    set={
        "IS_CURRENT": "'N'",               # mark old record as not current
        "END_DATE": "current_timestamp()"  # set end date to now
    }
).execute()

In [0]:
changed_insert_df = (
    changed_df
    .withColumn("LOAD_DATE", F.current_timestamp())
    .withColumn("IS_CURRENT", F.lit("Y"))
    .withColumn("EFFECTIVE_DATE", F.current_timestamp())
    .withColumn("END_DATE", F.lit("9999-12-31").cast("timestamp"))
)

changed_insert_df.write.format("delta").mode("append").saveAsTable(ods_provider_table)


In [0]:
new_provider_df = (
    new_provider_df
    .withColumn("LOAD_DATE", F.current_timestamp())
    .withColumn("IS_CURRENT", F.lit("Y"))
    .withColumn("EFFECTIVE_DATE", F.current_timestamp())
    .withColumn("END_DATE", F.lit("9999-12-31").cast("timestamp"))
)

new_provider_df.write.format("delta").mode("append").saveAsTable(ods_provider_table)


In [0]:
display(
    spark.sql(f"""
    SELECT *
    FROM {ods_provider_table}
    ORDER BY PROVIDER_ID
    """)
)
